In [2]:
from collections import defaultdict
from time import perf_counter
import os

try:
    import psutil

    process = psutil.Process(os.getpid())
    PSUTIL_AVAILABLE = True

except ImportError:
    process = None
    PSUTIL_AVAILABLE = False


# ============================================================
# GLOBAL CACHE
# ============================================================

kap_cache = {}

_NOT_CACHED = object()


# ============================================================
# BITMASK HELPERS
# ============================================================

def value_bit(x: int) -> int:
    """
    Convert integer x to its bit position.

    Example:
        1 -> 0001
        2 -> 0010
        3 -> 0100
        4 -> 1000
    """
    return 1 << (x - 1)


# ============================================================
# STATISTICS
# ============================================================

def make_depth_stats():
    return defaultdict(
        lambda: {
            "nodes": 0,
            "cache_hits": 0,
            "cache_misses": 0,
            "choices_tried": 0,
            "ap_prunes": 0,
            "lookahead_prunes": 0,
            "choices_survived": 0,
            "candidates_yielded": 0,
        }
    )


# ============================================================
# k-TERM ARITHMETIC PROGRESSION TEST
# ============================================================

def creates_kAP(
    current_mask: int,
    x: int,
    k: int,
    cache_stats: dict,
    depth_stats,
    depth: int,
    min_value: int = 1
) -> bool:
    """
    Determine whether adding x to the current set creates
    a k-term arithmetic progression.

    Because values are added in increasing order, x must be
    the largest/final value of any newly created progression.
    """

    key = (current_mask, x)

    cached_result = kap_cache.get(
        key,
        _NOT_CACHED
    )

    # --------------------------------------------------------
    # CACHE HIT
    # --------------------------------------------------------

    if cached_result is not _NOT_CACHED:

        cache_stats["hits"] += 1
        depth_stats[depth]["cache_hits"] += 1

        if cached_result:
            cache_stats["true_hits"] += 1
        else:
            cache_stats["false_hits"] += 1

        return cached_result

    # --------------------------------------------------------
    # CACHE MISS
    # --------------------------------------------------------

    cache_stats["misses"] += 1
    depth_stats[depth]["cache_misses"] += 1

    d_max = (x - min_value) // (k - 1)

    # --------------------------------------------------------
    # CHECK POSSIBLE COMMON DIFFERENCES
    # --------------------------------------------------------

    for d in range(1, d_max + 1):

        progression_found = True

        for i in range(1, k):

            previous_value = x - i * d

            bit = value_bit(previous_value)

            if not (current_mask & bit):
                progression_found = False
                break

        if progression_found:

            kap_cache[key] = True
            cache_stats["true_stores"] += 1

            return True

    # --------------------------------------------------------
    # NO PROGRESSION FOUND
    # --------------------------------------------------------

    kap_cache[key] = False
    cache_stats["false_stores"] += 1

    return False


# ============================================================
# BACKTRACKING SEARCH
# ============================================================

def generate_AP_free_candidates(
    S,
    size: int,
    k: int,
    stats: dict,
    cache_stats: dict,
    depth_stats
):
    """
    Generate AP-free subsets of S having exactly 'size'
    elements.

    Branches are pruned when:
      1. Adding x immediately creates a k-AP.
      2. Look-ahead shows that too few future values remain
         individually eligible to reach the target size.
    """

    S = list(S)

    if not S:
        return

    min_value = S[0]

    def backtrack(
        start: int,
        current: list,
        current_mask: int
    ):

        depth = len(current)

        stats["nodes"] += 1
        depth_stats[depth]["nodes"] += 1

        # ----------------------------------------------------
        # TARGET SIZE REACHED
        # ----------------------------------------------------

        if depth == size:

            stats["candidates_yielded"] += 1
            depth_stats[depth]["candidates_yielded"] += 1

            yield tuple(current)
            return

        # ----------------------------------------------------
        # TRY EACH POSSIBLE NEXT VALUE
        # ----------------------------------------------------

        for i in range(start, len(S)):

            x = S[i]

            stats["choices_tried"] += 1
            depth_stats[depth]["choices_tried"] += 1

            # ------------------------------------------------
            # IMMEDIATE AP PRUNE
            # ------------------------------------------------

            if creates_kAP(
                current_mask,
                x,
                k,
                cache_stats,
                depth_stats,
                depth,
                min_value
            ):

                stats["ap_prunes"] += 1
                depth_stats[depth]["ap_prunes"] += 1

                continue

            # ------------------------------------------------
            # TEMPORARILY ADD x USING BITMASK
            # ------------------------------------------------

            test_mask = (
                current_mask
                | value_bit(x)
            )

            needed_after_x = (
                size - (depth + 1)
            )

            eligible_after_x = 0

            # ------------------------------------------------
            # LOOK-AHEAD PRUNING
            # ------------------------------------------------

            if needed_after_x > 0:

                for j in range(
                    i + 1,
                    len(S)
                ):

                    y = S[j]

                    if not creates_kAP(
                        test_mask,
                        y,
                        k,
                        cache_stats,
                        depth_stats,
                        depth,
                        min_value
                    ):

                        eligible_after_x += 1

                        # We only need to know whether
                        # enough eligible values exist.
                        if (
                            eligible_after_x
                            >= needed_after_x
                        ):
                            break

            # ------------------------------------------------
            # NOT ENOUGH ELIGIBLE FUTURE VALUES
            # ------------------------------------------------

            if (
                eligible_after_x
                < needed_after_x
            ):

                stats["lookahead_prunes"] += 1
                depth_stats[depth]["lookahead_prunes"] += 1

                continue

            # ------------------------------------------------
            # BRANCH SURVIVES
            # ------------------------------------------------

            stats["choices_survived"] += 1
            depth_stats[depth]["choices_survived"] += 1

            current.append(x)

            yield from backtrack(
                i + 1,
                current,
                test_mask
            )

            current.pop()

    yield from backtrack(
        0,
        [],
        0
    )


# ============================================================
# COMPUTE r_k(N)
# ============================================================

def r(
    N: int,
    k: int,
    previous_max: int
):
    """
    Given r_k(N-1) = previous_max, test whether r_k(N)
    increases by one.

    Since adding one new element to the universe can increase
    the maximum AP-free subset size by at most one:

        r_k(N) ∈ {previous_max, previous_max + 1}
    """

    stats = {
        "nodes": 0,
        "choices_tried": 0,
        "ap_prunes": 0,
        "lookahead_prunes": 0,
        "choices_survived": 0,
        "candidates_yielded": 0,
        "elapsed_seconds": 0.0,
    }

    cache_stats = {
        "hits": 0,
        "misses": 0,
        "true_hits": 0,
        "false_hits": 0,
        "true_stores": 0,
        "false_stores": 0,
    }

    depth_stats = make_depth_stats()

    target_size = previous_max + 1

    S = range(
        1,
        N + 1
    )

    start_time = perf_counter()

    # --------------------------------------------------------
    # STOP AT FIRST VALID TARGET-SIZE CANDIDATE
    # --------------------------------------------------------

    for candidate in generate_AP_free_candidates(
        S,
        target_size,
        k,
        stats,
        cache_stats,
        depth_stats
    ):

        stats["elapsed_seconds"] = (
            perf_counter()
            - start_time
        )

        return (
            target_size,
            candidate,
            stats,
            cache_stats,
            depth_stats
        )

    # --------------------------------------------------------
    # NO TARGET-SIZE CANDIDATE EXISTS
    # --------------------------------------------------------

    stats["elapsed_seconds"] = (
        perf_counter()
        - start_time
    )

    return (
        previous_max,
        None,
        stats,
        cache_stats,
        depth_stats
    )


# ============================================================
# REPORTING
# ============================================================

def print_result(
    N,
    r_value,
    stats
):

    print()
    print(f"RESULT FOR N = {N}")
    print()

    header = (
        f"{'N':>3} | "
        f"{'r_k(N)':>6} | "
        f"{'Seconds':>10} | "
        f"{'Nodes':>12} | "
        f"{'Tried':>12} | "
        f"{'AP Prunes':>12} | "
        f"{'LA Prunes':>12} | "
        f"{'Survived':>12} | "
        f"{'Yielded':>8}"
    )

    print(header)
    print("-" * len(header))

    print(
        f"{N:>3} | "
        f"{r_value:>6} | "
        f"{stats['elapsed_seconds']:>10.4f} | "
        f"{stats['nodes']:>12,} | "
        f"{stats['choices_tried']:>12,} | "
        f"{stats['ap_prunes']:>12,} | "
        f"{stats['lookahead_prunes']:>12,} | "
        f"{stats['choices_survived']:>12,} | "
        f"{stats['candidates_yielded']:>8,}"
    )


def print_cache_summary(
    cache_stats
):

    hits = cache_stats["hits"]
    misses = cache_stats["misses"]

    true_hits = cache_stats["true_hits"]
    false_hits = cache_stats["false_hits"]

    true_stores = cache_stats["true_stores"]
    false_stores = cache_stats["false_stores"]

    total_calls = (
        hits + misses
    )

    hit_rate = (
        hits / total_calls
        if total_calls
        else 0
    )

    true_reuse_rate = (
        true_hits / true_stores
        if true_stores
        else 0
    )

    false_reuse_rate = (
        false_hits / false_stores
        if false_stores
        else 0
    )

    print()

    print(f"Cache Hits:          {hits:,}")
    print(f"Cache Misses:        {misses:,}")
    print(f"Cache Hit Rate:      {hit_rate:.2%}")

    print()

    print(f"True Cache Hits:     {true_hits:,}")
    print(f"False Cache Hits:    {false_hits:,}")

    print()

    print(f"True Stores:         {true_stores:,}")
    print(f"False Stores:        {false_stores:,}")

    print()

    print(
        f"True Reuse Ratio:    "
        f"{true_reuse_rate:.3f}"
    )

    print(
        f"False Reuse Ratio:   "
        f"{false_reuse_rate:.3f}"
    )

    print()

    print(
        f"Cache Entries:       "
        f"{len(kap_cache):,}"
    )

    if PSUTIL_AVAILABLE:

        memory_gb = (
            process.memory_info().rss
            / (1024 ** 3)
        )

        print(
            f"Process Memory:      "
            f"{memory_gb:.3f} GB"
        )


def print_depth_stats(
    N,
    target_size,
    depth_stats
):

    if not depth_stats:
        return

    max_node_depth = max(
        depth_stats,
        key=lambda depth:
            depth_stats[depth]["nodes"]
    )

    max_nodes = (
        depth_stats[
            max_node_depth
        ]["nodes"]
    )

    print()

    print(
        f"Max node depth: "
        f"{max_node_depth} "
        f"({max_nodes:,} nodes)"
    )

    print()

    print(
        f"Depth statistics for N = {N}, "
        f"searching for subset size "
        f"{target_size}"
    )

    print()

    header = (
        f"{'Depth':>5} | "
        f"{'Nodes':>12} | "
        f"{'Cache Hits':>12} | "
        f"{'Cache Misses':>12} | "
        f"{'Tried':>12} | "
        f"{'AP Prunes':>12} | "
        f"{'LA Prunes':>12} | "
        f"{'Survived':>12} | "
        f"{'Yielded':>8}"
    )

    print(header)
    print("-" * len(header))

    for depth in sorted(
        depth_stats
    ):

        ds = depth_stats[depth]

        print(
            f"{depth:>5} | "
            f"{ds['nodes']:>12,} | "
            f"{ds['cache_hits']:>12,} | "
            f"{ds['cache_misses']:>12,} | "
            f"{ds['choices_tried']:>12,} | "
            f"{ds['ap_prunes']:>12,} | "
            f"{ds['lookahead_prunes']:>12,} | "
            f"{ds['choices_survived']:>12,} | "
            f"{ds['candidates_yielded']:>8,}"
        )

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

k = 3

N_start = 5
N_end = 20

# Must be the known exact value r_k(N_start)
max_size = 4

# Clear cache between each N so memory does not accumulate
# indefinitely across the entire run.
CLEAR_CACHE_EACH_N = True


# ============================================================
# RUN
# ============================================================

print(
    f"N = {N_start}, "
    f"r_{k}({N_start}) = {max_size}"
)

print("=" * 130)


for N in range(
    N_start + 1,
    N_end + 1
):

    if CLEAR_CACHE_EACH_N:
        kap_cache.clear()

    previous_max = max_size

    (
        max_size,
        candidate,
        stats,
        cache_stats,
        depth_stats
    ) = r(
        N,
        k,
        previous_max
    )

    print_result(
        N,
        max_size,
        stats
    )

    print_cache_summary(
        cache_stats
    )

    target_size = (
        previous_max + 1
    )

    print_depth_stats(
        N,
        target_size,
        depth_stats
    )

    if candidate is not None:

        print()
        print(
            f"Candidate: "
            f"{candidate}"
        )

    print()
    print("=" * 130)

N = 5, r_3(5) = 4

RESULT FOR N = 6

  N | r_k(N) |    Seconds |        Nodes |        Tried |    AP Prunes |    LA Prunes |     Survived |  Yielded
---------------------------------------------------------------------------------------------------------------
  6 |      4 |     0.0001 |            4 |           19 |            1 |           15 |            3 |        0

Cache Hits:          12
Cache Misses:        40
Cache Hit Rate:      23.08%

True Cache Hits:     1
False Cache Hits:    11

True Stores:         5
False Stores:        35

True Reuse Ratio:    0.200
False Reuse Ratio:   0.314

Cache Entries:       40
Process Memory:      0.300 GB

Max node depth: 1 (2 nodes)

Depth statistics for N = 6, searching for subset size 5

Depth |        Nodes |   Cache Hits | Cache Misses |        Tried |    AP Prunes |    LA Prunes |     Survived |  Yielded
-------------------------------------------------------------------------------------------------------------------------
    0 |      